# Shopify Sales Data Analytics
## Complete EDA, Data Cleaning, Feature Engineering & Star Schema Creation
---
**Dataset:** Shopify Sales.xlsx — 7,431 line-item transactions (Mar 18–24, 2025)  
**19 Columns:** Order details, customer info, product info, geography, payment, financials

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

print('Libraries loaded successfully.')

## 2. Load Raw Data

In [ ]:
# Load from raw folder
raw_path = '../data/raw/Shopify Sales.xlsx'
df_raw = pd.read_excel(raw_path, sheet_name='shopify_sales')
print(f'Raw shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# Column info
df_raw.info()

In [ ]:
# Statistical summary
df_raw.describe(include='all')

## 3. Data Quality Assessment

In [ ]:
# Missing values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print('Missing Values:')
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Duplicate check
dup_all = df_raw.duplicated().sum()
dup_line_item = df_raw['Admin Graphql Api Id'].duplicated().sum()
print(f'Fully duplicate rows: {dup_all}')
print(f'Duplicate line item IDs: {dup_line_item}')

In [ ]:
# Data types
print('Data types:')
print(df_raw.dtypes)

In [ ]:
# Unique value counts for categorical columns
cat_cols = ['Billing Address Country', 'Currency', 'Gateway', 'Product Type', 'Billing Address Province']
for col in cat_cols:
    print(f'{col}: {df_raw[col].nunique()} unique — {sorted(df_raw[col].dropna().unique().tolist())}')
    print()

In [ ]:
# Outlier analysis using IQR on financial columns
financial_cols = ['Quantity', 'Subtotal Price', 'Total Price Usd', 'Total Tax']
for col in financial_cols:
    q1 = df_raw[col].quantile(0.25)
    q3 = df_raw[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 3 * iqr
    upper = q3 + 3 * iqr
    outliers = df_raw[(df_raw[col] < lower) | (df_raw[col] > upper)]
    print(f'{col}: IQR bounds [{lower:.2f}, {upper:.2f}] — {len(outliers)} outliers ({len(outliers)/len(df_raw)*100:.2f}%)')

## 4. Data Cleaning

In [ ]:
df = df_raw.copy()

# --- 4.1 Standardise column names ---
df.columns = [
    'line_item_id', 'order_number', 'country', 'first_name', 'last_name',
    'province', 'zip_code', 'city', 'currency', 'customer_id',
    'invoice_date', 'gateway', 'product_id', 'product_type',
    'variant_id', 'quantity', 'subtotal_price', 'total_price_usd', 'total_tax'
]

print('Columns renamed:', df.columns.tolist())

In [ ]:
# --- 4.2 Remove fully duplicate rows ---
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed: {before - len(df)} rows')

In [ ]:
# --- 4.3 Fix data types ---
# Date: parse invoice_date
df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')

# Numeric: ensure correct types
for col in ['quantity', 'subtotal_price', 'total_price_usd', 'total_tax']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# IDs: keep as strings
df['customer_id'] = df['customer_id'].astype(str)
df['product_id'] = df['product_id'].astype(str)
df['variant_id'] = df['variant_id'].astype(str)
df['order_number'] = df['order_number'].astype(str)

print('Date range after parse:', df['invoice_date'].min(), 'to', df['invoice_date'].max())
print('NaT dates:', df['invoice_date'].isna().sum())

In [ ]:
# --- 4.4 Handle missing values ---
# product_id (11 missing) and variant_id (4 missing) — impute with 'UNKNOWN'
df['product_id'].fillna('UNKNOWN', inplace=True)
df['variant_id'].fillna('UNKNOWN', inplace=True)

# Drop rows where financial data is null
before = len(df)
df.dropna(subset=['subtotal_price', 'total_price_usd', 'total_tax', 'invoice_date'], inplace=True)
print(f'Rows dropped due to null financials/dates: {before - len(df)}')

In [ ]:
# --- 4.5 Standardise string/category values ---
df['city'] = df['city'].str.strip().str.upper()
df['province'] = df['province'].str.strip().str.title()
df['country'] = df['country'].str.strip().str.title()
df['gateway'] = df['gateway'].str.strip().str.lower()
df['product_type'] = df['product_type'].str.strip().str.title()
df['currency'] = df['currency'].str.strip().str.upper()

# Fix known inconsistency — "Boy'S" → "Boy's Shoes"
df['product_type'] = df['product_type'].replace({"Boy'S": "Boy's Shoes", "Boy's": "Boy's Shoes"})

# Clean gateway names for display
gateway_map = {
    'shopify_payments': 'Shopify Payments',
    'paypal': 'PayPal',
    'amazon_payments': 'Amazon Pay',
    'gift_card': 'Gift Card',
    'manual': 'Manual'
}
df['gateway'] = df['gateway'].map(gateway_map).fillna(df['gateway'])

print('Product types:', sorted(df['product_type'].unique()))
print('Gateways:', sorted(df['gateway'].unique()))

In [ ]:
# --- 4.6 Validate numeric bounds (remove invalid outliers) ---
# Remove rows with quantity < 1 or > 20 (business rule)
before = len(df)
df = df[(df['quantity'] >= 1) & (df['quantity'] <= 20)]
print(f'Invalid quantity rows removed: {before - len(df)}')

# Remove rows with zero or negative prices
before = len(df)
df = df[(df['total_price_usd'] > 0) & (df['subtotal_price'] > 0)]
print(f'Non-positive price rows removed: {before - len(df)}')

In [ ]:
# --- 4.7 Feature Engineering ---
df['order_date'] = df['invoice_date'].dt.date
df['order_year'] = df['invoice_date'].dt.year
df['order_month'] = df['invoice_date'].dt.month
df['order_month_name'] = df['invoice_date'].dt.strftime('%B')
df['order_week'] = df['invoice_date'].dt.isocalendar().week.astype(int)
df['order_day'] = df['invoice_date'].dt.day
df['day_of_week'] = df['invoice_date'].dt.day_name()
df['hour'] = df['invoice_date'].dt.hour
df['unit_price'] = (df['subtotal_price'] / df['quantity']).round(2)
df['tax_rate'] = (df['total_tax'] / df['subtotal_price'] * 100).round(2)
df['customer_full_name'] = df['first_name'] + ' ' + df['last_name']

print('Features added. Final shape:', df.shape)
df.head(3)

In [ ]:
# Save cleaned dataset
os.makedirs('../data/cleaned', exist_ok=True)
df.to_csv('../data/cleaned/shopify_sales_cleaned.csv', index=False)
print('Cleaned dataset saved.')

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# --- KPI Summary ---
total_revenue = df['total_price_usd'].sum()
total_orders = df['order_number'].nunique()
total_customers = df['customer_id'].nunique()
total_items = df['quantity'].sum()
avg_order_value = df.groupby('order_number')['total_price_usd'].sum().mean()
avg_unit_price = df['unit_price'].mean()
total_tax = df['total_tax'].sum()

print(f'=== KEY PERFORMANCE INDICATORS ===')
print(f'Total Revenue (USD):     ${total_revenue:,.2f}')
print(f'Total Orders:            {total_orders:,}')
print(f'Total Customers:         {total_customers:,}')
print(f'Total Items Sold:        {total_items:,}')
print(f'Avg Order Value (USD):   ${avg_order_value:,.2f}')
print(f'Avg Unit Price (USD):    ${avg_unit_price:,.2f}')
print(f'Total Tax Collected:     ${total_tax:,.2f}')

In [ ]:
# --- Revenue by Product Type ---
rev_product = df.groupby('product_type').agg(
    total_revenue=('total_price_usd', 'sum'),
    total_orders=('order_number', 'nunique'),
    total_qty=('quantity', 'sum')
).sort_values('total_revenue', ascending=False).reset_index()

fig = px.bar(rev_product, x='product_type', y='total_revenue',
             title='Revenue by Product Type', color='total_revenue',
             color_continuous_scale='Blues', labels={'total_revenue': 'Revenue (USD)'})
fig.update_layout(xaxis_tickangle=-30)
fig.show()
print(rev_product.to_string())

In [ ]:
# --- Daily Revenue Trend ---
daily = df.groupby('order_date')['total_price_usd'].sum().reset_index()
daily.columns = ['date', 'revenue']

fig = px.line(daily, x='date', y='revenue', title='Daily Revenue Trend (Mar 18–24, 2025)',
              markers=True, labels={'revenue': 'Revenue (USD)'})
fig.update_traces(line_color='#1F77B4', line_width=2.5)
fig.show()

In [ ]:
# --- Revenue by Gateway ---
rev_gateway = df.groupby('gateway')['total_price_usd'].sum().sort_values(ascending=False).reset_index()

fig = px.pie(rev_gateway, names='gateway', values='total_price_usd',
             title='Revenue Share by Payment Gateway', hole=0.4)
fig.show()
print(rev_gateway)

In [ ]:
# --- Revenue by Province (Top 15) ---
rev_province = df.groupby('province')['total_price_usd'].sum().sort_values(ascending=False).head(15).reset_index()

fig = px.bar(rev_province, x='total_price_usd', y='province', orientation='h',
             title='Top 15 States by Revenue', color='total_price_usd',
             color_continuous_scale='Viridis')
fig.show()

In [ ]:
# --- Quantity Distribution ---
qty_dist = df['quantity'].value_counts().sort_index()
fig = px.bar(x=qty_dist.index, y=qty_dist.values, title='Order Quantity Distribution',
             labels={'x': 'Quantity', 'y': 'Count'})
fig.show()

In [ ]:
# --- Revenue by Day of Week ---
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
rev_dow = df.groupby('day_of_week')['total_price_usd'].sum().reindex(dow_order).reset_index()
fig = px.bar(rev_dow, x='day_of_week', y='total_price_usd',
             title='Revenue by Day of Week', color='total_price_usd',
             color_continuous_scale='Blues')
fig.show()

In [ ]:
# --- Revenue by Hour of Day ---
rev_hour = df.groupby('hour')['total_price_usd'].sum().reset_index()
fig = px.line(rev_hour, x='hour', y='total_price_usd', markers=True,
              title='Revenue by Hour of Day', labels={'hour': 'Hour', 'total_price_usd': 'Revenue (USD)'})
fig.show()

In [ ]:
# --- Top 10 Customers by Revenue ---
top_customers = df.groupby(['customer_id', 'customer_full_name']).agg(
    total_revenue=('total_price_usd', 'sum'),
    total_orders=('order_number', 'nunique')
).sort_values('total_revenue', ascending=False).head(10).reset_index()
print('Top 10 Customers:')
print(top_customers.to_string())

In [ ]:
# --- Price distribution by product type ---
fig = px.box(df, x='product_type', y='unit_price', title='Unit Price Distribution by Product Type',
             color='product_type')
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

In [ ]:
# --- Correlation heatmap ---
num_cols = ['quantity', 'subtotal_price', 'total_price_usd', 'total_tax', 'unit_price', 'tax_rate']
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
plt.title('Correlation Heatmap — Numeric Variables')
plt.tight_layout()
plt.show()

## 6. Star Schema Construction

In [ ]:
os.makedirs('../data/star_schema', exist_ok=True)

# ---- DimDate ----
date_range = pd.date_range(start=df['invoice_date'].min().date(),
                           end=df['invoice_date'].max().date(), freq='D')
dim_date = pd.DataFrame({'full_date': date_range})
dim_date['date_key'] = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['month_name'] = dim_date['full_date'].dt.strftime('%B')
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['quarter_name'] = 'Q' + dim_date['quarter'].astype(str)
dim_date['week'] = dim_date['full_date'].dt.isocalendar().week.astype(int)
dim_date['day'] = dim_date['full_date'].dt.day
dim_date['day_of_week_num'] = dim_date['full_date'].dt.dayofweek
dim_date['day_name'] = dim_date['full_date'].dt.day_name()
dim_date['is_weekend'] = dim_date['day_of_week_num'].isin([5, 6]).astype(int)

dim_date.to_csv('../data/star_schema/DimDate.csv', index=False)
print(f'DimDate: {dim_date.shape}')
dim_date.head()

In [ ]:
# ---- DimProduct ----
dim_product = df[['product_id', 'product_type', 'variant_id']].drop_duplicates(subset=['product_id', 'variant_id'])
dim_product = dim_product.reset_index(drop=True)
dim_product.insert(0, 'product_key', range(1, len(dim_product) + 1))
dim_product['product_category'] = dim_product['product_type'].apply(
    lambda x: 'Footwear' if any(w in x for w in ['Shoe', 'Boot', 'Sandal', 'Flip', 'Clog', 'Water']) else
              ('Apparel' if x == 'Jackets' else
               ('Gift' if x == 'Gift Card' else 'Other'))
)

dim_product.to_csv('../data/star_schema/DimProduct.csv', index=False)
print(f'DimProduct: {dim_product.shape}')
dim_product.head()

In [ ]:
# ---- DimCustomer ----
dim_customer = df[['customer_id', 'first_name', 'last_name', 'customer_full_name']].drop_duplicates(subset=['customer_id'])
dim_customer = dim_customer.reset_index(drop=True)
dim_customer.insert(0, 'customer_key', range(1, len(dim_customer) + 1))

dim_customer.to_csv('../data/star_schema/DimCustomer.csv', index=False)
print(f'DimCustomer: {dim_customer.shape}')
dim_customer.head()

In [ ]:
# ---- DimLocation ----
dim_location = df[['city', 'province', 'country', 'zip_code']].drop_duplicates(subset=['city', 'province'])
dim_location = dim_location.reset_index(drop=True)
dim_location.insert(0, 'location_key', range(1, len(dim_location) + 1))
dim_location['region'] = dim_location['province'].map({
    'California': 'West', 'Oregon': 'West', 'Washington': 'West', 'Nevada': 'West',
    'Arizona': 'West', 'Utah': 'West', 'Colorado': 'West', 'Montana': 'West',
    'Idaho': 'West', 'Wyoming': 'West', 'Alaska': 'West', 'Hawaii': 'West',
    'New Mexico': 'West',
    'Texas': 'South', 'Florida': 'South', 'Georgia': 'South', 'North Carolina': 'South',
    'South Carolina': 'South', 'Virginia': 'South', 'Tennessee': 'South', 'Alabama': 'South',
    'Mississippi': 'South', 'Arkansas': 'South', 'Louisiana': 'South', 'Oklahoma': 'South',
    'Kentucky': 'South', 'West Virginia': 'South', 'Maryland': 'South',
    'District Of Columbia': 'South', 'Delaware': 'South',
    'New York': 'Northeast', 'Pennsylvania': 'Northeast', 'New Jersey': 'Northeast',
    'Massachusetts': 'Northeast', 'Connecticut': 'Northeast', 'Rhode Island': 'Northeast',
    'New Hampshire': 'Northeast', 'Vermont': 'Northeast', 'Maine': 'Northeast',
    'Illinois': 'Midwest', 'Ohio': 'Midwest', 'Michigan': 'Midwest', 'Indiana': 'Midwest',
    'Wisconsin': 'Midwest', 'Minnesota': 'Midwest', 'Iowa': 'Midwest', 'Missouri': 'Midwest',
    'North Dakota': 'Midwest', 'South Dakota': 'Midwest', 'Nebraska': 'Midwest',
    'Kansas': 'Midwest',
}).fillna('Other')

dim_location.to_csv('../data/star_schema/DimLocation.csv', index=False)
print(f'DimLocation: {dim_location.shape}')
dim_location.head()

In [ ]:
# ---- DimPayment ----
dim_payment = df[['gateway']].drop_duplicates().reset_index(drop=True)
dim_payment.insert(0, 'payment_key', range(1, len(dim_payment) + 1))
dim_payment.columns = ['payment_key', 'gateway']
payment_type_map = {
    'Shopify Payments': 'Digital Wallet',
    'PayPal': 'Digital Wallet',
    'Amazon Pay': 'Digital Wallet',
    'Gift Card': 'Store Credit',
    'Manual': 'Manual'
}
dim_payment['payment_type'] = dim_payment['gateway'].map(payment_type_map).fillna('Other')
dim_payment['is_online'] = (dim_payment['gateway'] != 'Manual').astype(int)

dim_payment.to_csv('../data/star_schema/DimPayment.csv', index=False)
print(f'DimPayment: {dim_payment.shape}')
print(dim_payment)

In [ ]:
# ---- FactSales ----
# Build date_key from invoice_date
df['date_key'] = df['invoice_date'].dt.strftime('%Y%m%d').astype(int)

# Merge dimension keys
fact = df.merge(dim_product[['product_key', 'product_id', 'variant_id']], on=['product_id', 'variant_id'], how='left')
fact = fact.merge(dim_customer[['customer_key', 'customer_id']], on='customer_id', how='left')
fact = fact.merge(dim_location[['location_key', 'city', 'province']], on=['city', 'province'], how='left')
fact = fact.merge(dim_payment[['payment_key', 'gateway']], on='gateway', how='left')

# Select FactSales columns
fact_sales = fact[[
    'line_item_id', 'order_number', 'date_key',
    'product_key', 'customer_key', 'location_key', 'payment_key',
    'quantity', 'unit_price', 'subtotal_price', 'total_price_usd', 'total_tax', 'tax_rate'
]].copy()

fact_sales.to_csv('../data/star_schema/FactSales.csv', index=False)
print(f'FactSales: {fact_sales.shape}')
fact_sales.head()

In [ ]:
# Verify referential integrity
print('Null product_key:', fact_sales['product_key'].isna().sum())
print('Null customer_key:', fact_sales['customer_key'].isna().sum())
print('Null location_key:', fact_sales['location_key'].isna().sum())
print('Null payment_key:', fact_sales['payment_key'].isna().sum())
print('Null date_key:', fact_sales['date_key'].isna().sum())
print('\nStar Schema grain: One row = one line item on an order')

## 7. Star Schema Diagram

```
                    DimDate
                  (date_key PK)
                       |
DimCustomer ──── FactSales ──── DimProduct
(cust_key PK)  (line_item_id PK) (prod_key PK)
                       |
          DimLocation ─┼─ DimPayment
         (loc_key PK)     (pay_key PK)
```

**Grain:** One row per line item (individual product on an order)  
**Fact measures:** quantity, unit_price, subtotal_price, total_price_usd, total_tax, tax_rate

In [ ]:
print('=== PROJECT SUMMARY ===')
print(f'Raw rows: 7,431')
print(f'Cleaned rows: {len(df)}')
print(f'Star Schema tables: FactSales, DimDate, DimProduct, DimCustomer, DimLocation, DimPayment')
print(f'FactSales rows: {len(fact_sales)}')
print(f'Total Revenue: ${df["total_price_usd"].sum():,.2f}')
print(f'Date Range: {df["invoice_date"].min()} → {df["invoice_date"].max()}')
print(f'Unique Products: {df["product_type"].nunique()}')
print(f'Unique Customers: {df["customer_id"].nunique()}')
print(f'Unique States: {df["province"].nunique()}')
print(f'Unique Cities: {df["city"].nunique()}')